In [46]:
import numpy as np
import pandas as pd
import zipfile
import os
import glob

In [47]:
# Extract FY2021 and FY2022 data files
fy2021_2022_files = ['TRK_13139_FY2021.zip', 'TRK_13139_FY2022.zip']

for zip_file in fy2021_2022_files:
    if os.path.exists(zip_file):
        print(f"Extracting {zip_file}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall('.')
            print(f"  Extracted: {zip_ref.namelist()}")
    else:
        print(f"File not found: {zip_file}")

print("FY2021 and FY2022 data extraction complete!")

Extracting TRK_13139_FY2021.zip...
  Extracted: ['TRK_13139_FY2021.csv']
Extracting TRK_13139_FY2022.zip...
  Extracted: ['TRK_13139_FY2022.csv']
FY2021 and FY2022 data extraction complete!


In [48]:
# Extract FY2024 data files
fy2024_files = ['TRK_13139_FY2024_multi_reg.zip', 'TRK_13139_FY2024_single_reg.zip']

for zip_file in fy2024_files:
    if os.path.exists(zip_file):
        print(f"Extracting {zip_file}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall('.')
            print(f"  Extracted: {zip_ref.namelist()}")
    else:
        print(f"File not found: {zip_file}")

print("FY2024 data extraction complete!")

Extracting TRK_13139_FY2024_multi_reg.zip...
  Extracted: ['TRK_13139_FY2024_multi_reg.csv']
Extracting TRK_13139_FY2024_single_reg.zip...
  Extracted: ['TRK_13139_FY2024_single_reg.csv']
FY2024 data extraction complete!


In [49]:
# Combine split zip files and extract
split_files = sorted(glob.glob('TRK_13139_FY2023.zip.*'))
print(f"Found {len(split_files)} split files: {split_files}")

# Combine the split files into a single zip file
output_zip = 'TRK_13139_FY2023.zip'
with open(output_zip, 'wb') as outfile:
    for split_file in split_files:
        print(f"Processing {split_file}...")
        with open(split_file, 'rb') as infile:
            outfile.write(infile.read())

print(f"Combined zip file created: {output_zip}")

# Extract the combined zip file
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall('.')
    print(f"Extracted files: {zip_ref.namelist()}")

print("Data extraction complete!")

Found 3 split files: ['TRK_13139_FY2023.zip.001', 'TRK_13139_FY2023.zip.002', 'TRK_13139_FY2023.zip.003']
Processing TRK_13139_FY2023.zip.001...
Processing TRK_13139_FY2023.zip.002...
Processing TRK_13139_FY2023.zip.003...
Combined zip file created: TRK_13139_FY2023.zip
Extracted files: ['TRK_13139_FY2023.csv']
Data extraction complete!


In [50]:
# 合并所有数据（直接合并，不添加额外列）
df_all = pd.concat([df_2021, df_2022, df_2023, df_2024_multi, df_2024_single], 
                    ignore_index=True)

print(f"合并后的数据形状: {df_all.shape}")
print(f"总行数: {df_all.shape[0]:,}")
print(f"总列数: {df_all.shape[1]}")

合并后的数据形状: (1804286, 58)
总行数: 1,804,286
总列数: 58


In [51]:
# 转置显示前几条记录，这样可以看到所有列
print("前3条记录的所有列（转置显示）:")
print("="*80)
df_all.head(3).T

前3条记录的所有列（转置显示）:


,0,1,2
bcn,(b)(6),(b)(6),(b)(6)
country_of_birth,CHN,IND,CAN
country_of_nationality,CHN,IND,CAN
ben_date_of_birth,(b)(6),(b)(6),(b)(6)
ben_year_of_birth,1981,1994,1988
gender,male,male,male
employer_name,D&R I.P. Law Firm,ITTECHNICA INC,"Tesla, Inc."
FEIN,453745389,824530582,912197729
mail_addr,108 N Ynez Ave,1825 W Walnut Hill Ln,3500 Deer Creek Rd
city,Monterey Park,Irving,Palo Alto


# 数据列详细分析

分析5个最重要的列：
1. **Status_type** - 申请状态类型
2. **FIRST_DECISION** - 首次决策结果
3. **I129_employer_name** - 雇主名称
4. **JOB_TITLE** - 职位名称
5. **BEN_PFIELD_OF_STUDY** - 受益人学习领域

In [52]:
# 1. Status_type
print("Status_type")

# Datatype
print(f"\n【Datatype】")
print(f"Type: {df_all['status_type'].dtype}")
print('Datatype: Category')

# 基本统计
print(f"\n【Basci statistics】")
print(f"Total rows: {len(df_all):,}")
print(f"None-null values: {df_all['status_type'].notna().sum():,}")
print(f"Missing values: {df_all['status_type'].isna().sum():,}")
print(f"Missing portion: {df_all['status_type'].isna().sum() / len(df_all) * 100:.2f}%")
print(f"Unique values: {df_all['status_type'].nunique()}")

# 前5个最常见的值
print(f"\n【Top 5 values】")
top5 = df_all['status_type'].value_counts().head(5)
for value, count in top5.items():
    percentage = count / len(df_all) * 100
    print(f"  {value}: {count:,} ({percentage:.2f}%)")

# 详细信息
print(f"\n【Describe】")
df_all['status_type'].describe()
df_all['status_type'].info()

Status_type

【Datatype】
Type: object
Datatype: Category

【Basci statistics】
Total rows: 1,804,286
None-null values: 1,804,286
Missing values: 0
Missing portion: 0.00%
Unique values: 4

【Top 5 values】
  ELIGIBLE: 1,086,947 (60.24%)
  SELECTED: 572,191 (31.71%)
  CREATED: 145,009 (8.04%)
  (b)(3) (b)(6) (b)(7)(c): 139 (0.01%)

【Describe】
<class 'pandas.core.series.Series'>
RangeIndex: 1804286 entries, 0 to 1804285
Series name: status_type
Non-Null Count    Dtype 
--------------    ----- 
1804286 non-null  object
dtypes: object(1)
memory usage: 13.8+ MB


In [53]:
# 2. FIRST_DECISION
print("FIRST_DECISION")

# Datatype
print(f"\n【Datatype】")
print(f"Type: {df_all['FIRST_DECISION'].dtype}")
print('Datatype: Category')

# 基本统计
print(f"\n【Basic statistics】")
print(f"Total rows: {len(df_all):,}")
print(f"Non-null values: {df_all['FIRST_DECISION'].notna().sum():,}")
print(f"Missing values: {df_all['FIRST_DECISION'].isna().sum():,}")
print(f"Missing portion: {df_all['FIRST_DECISION'].isna().sum() / len(df_all) * 100:.2f}%")
print(f"Unique values: {df_all['FIRST_DECISION'].nunique()}")

# 前5个最常见的值
print(f"\n【Top 5 values】")
top5 = df_all['FIRST_DECISION'].value_counts().head(5)
for value, count in top5.items():
    percentage = count / len(df_all) * 100
    print(f"  {value}: {count:,} ({percentage:.2f}%)")

# 详细信息
print(f"\n【Describe】")
df_all['FIRST_DECISION'].describe()
df_all['FIRST_DECISION'].info()

FIRST_DECISION

【Datatype】
Type: object
Datatype: Category

【Basic statistics】
Total rows: 1,804,286
Non-null values: 375,045
Missing values: 1,429,241
Missing portion: 79.21%
Unique values: 3

【Top 5 values】
  Approved: 363,843 (20.17%)
  Denied: 11,063 (0.61%)
  (b)(3) (b)(6) (b)(7)(c): 139 (0.01%)

【Describe】
<class 'pandas.core.series.Series'>
RangeIndex: 1804286 entries, 0 to 1804285
Series name: FIRST_DECISION
Non-Null Count   Dtype 
--------------   ----- 
375045 non-null  object
dtypes: object(1)
memory usage: 13.8+ MB


In [54]:
# 3. I129_employer_name
print("i129_employer_name")

# Datatype
print(f"\n【Datatype】")
print(f"Type: {df_all['i129_employer_name'].dtype}")
print('Datatype: Category')

# 基本统计
print(f"\n【Basic statistics】")
print(f"Total rows: {len(df_all):,}")
print(f"Non-null values: {df_all['i129_employer_name'].notna().sum():,}")
print(f"Missing values: {df_all['i129_employer_name'].isna().sum():,}")
print(f"Missing portion: {df_all['i129_employer_name'].isna().sum() / len(df_all) * 100:.2f}%")
print(f"Unique values: {df_all['i129_employer_name'].nunique():,}")

# 前5个最常见的值
print(f"\n【Top 5 values】")
top5 = df_all['i129_employer_name'].value_counts().head(5)
for value, count in top5.items():
    percentage = count / len(df_all) * 100
    print(f"  {value}: {count:,} ({percentage:.2f}%)")

# 详细信息
print(f"\n【Describe】")
df_all['i129_employer_name'].describe()
df_all['i129_employer_name'].info()

i129_employer_name

【Datatype】
Type: object
Datatype: Category

【Basic statistics】
Total rows: 1,804,286
Non-null values: 376,610
Missing values: 1,427,676
Missing portion: 79.13%
Unique values: 56,652

【Top 5 values】
  INFOSYS LIMITED: 11,444 (0.63%)
  TATA CONSULTANCY SVCS LTD: 8,263 (0.46%)
  AMAZON COM SERVICES LLC: 7,114 (0.39%)
  AMAZON.COM SERVICES LLC: 6,542 (0.36%)
  GOOGLE LLC: 5,284 (0.29%)

【Describe】
<class 'pandas.core.series.Series'>
RangeIndex: 1804286 entries, 0 to 1804285
Series name: i129_employer_name
Non-Null Count   Dtype 
--------------   ----- 
376610 non-null  object
dtypes: object(1)
memory usage: 13.8+ MB


In [55]:
# 4. JOB_TITLE
print("JOB_TITLE")

# Datatype
print(f"\n【Datatype】")
print(f"Type: {df_all['JOB_TITLE'].dtype}")
print('Datatype: Category')

# 基本统计
print(f"\n【Basic statistics】")
print(f"Total rows: {len(df_all):,}")
print(f"Non-null values: {df_all['JOB_TITLE'].notna().sum():,}")
print(f"Missing values: {df_all['JOB_TITLE'].isna().sum():,}")
print(f"Missing portion: {df_all['JOB_TITLE'].isna().sum() / len(df_all) * 100:.2f}%")
print(f"Unique values: {df_all['JOB_TITLE'].nunique():,}")

# 前5个最常见的值
print(f"\n【Top 5 values】")
top5 = df_all['JOB_TITLE'].value_counts().head(5)
for value, count in top5.items():
    percentage = count / len(df_all) * 100
    print(f"  {value}: {count:,} ({percentage:.2f}%)")

# 详细信息
print(f"\n【Describe】")
df_all['JOB_TITLE'].describe()
df_all['JOB_TITLE'].info()

JOB_TITLE

【Datatype】
Type: object
Datatype: Category

【Basic statistics】
Total rows: 1,804,286
Non-null values: 247,953
Missing values: 1,556,333
Missing portion: 86.26%
Unique values: 55,278

【Top 5 values】
  SOFTWARE ENGINEER: 19,255 (1.07%)
  SOFTWARE DEVELOPER: 17,126 (0.95%)
  SOFTWARE DEVELOPMENT ENGINEER I: 4,484 (0.25%)
  ANALYST: 2,390 (0.13%)
  ARCHITECT: 2,213 (0.12%)

【Describe】
<class 'pandas.core.series.Series'>
RangeIndex: 1804286 entries, 0 to 1804285
Series name: JOB_TITLE
Non-Null Count   Dtype 
--------------   ----- 
247953 non-null  object
dtypes: object(1)
memory usage: 13.8+ MB


In [56]:
# 5. BEN_PFIELD_OF_STUDY
print("BEN_PFIELD_OF_STUDY")

# Datatype
print(f"\n【Datatype】")
print(f"Type: {df_all['BEN_PFIELD_OF_STUDY'].dtype}")
print('Datatype: Category')

# 基本统计
print(f"\n【Basic statistics】")
print(f"Total rows: {len(df_all):,}")
print(f"Non-null values: {df_all['BEN_PFIELD_OF_STUDY'].notna().sum():,}")
print(f"Missing values: {df_all['BEN_PFIELD_OF_STUDY'].isna().sum():,}")
print(f"Missing portion: {df_all['BEN_PFIELD_OF_STUDY'].isna().sum() / len(df_all) * 100:.2f}%")
print(f"Unique values: {df_all['BEN_PFIELD_OF_STUDY'].nunique():,}")

# 前5个最常见的值
print(f"\n【Top 5 values】")
top5 = df_all['BEN_PFIELD_OF_STUDY'].value_counts().head(5)
for value, count in top5.items():
    percentage = count / len(df_all) * 100
    print(f"  {value}: {count:,} ({percentage:.2f}%)")

# 详细信息
print(f"\n【Describe】")
df_all['BEN_PFIELD_OF_STUDY'].describe()
df_all['BEN_PFIELD_OF_STUDY'].info()

BEN_PFIELD_OF_STUDY

【Datatype】
Type: object
Datatype: Category

【Basic statistics】
Total rows: 1,804,286
Non-null values: 376,319
Missing values: 1,427,967
Missing portion: 79.14%
Unique values: 41,355

【Top 5 values】
  COMPUTER SCIENCE: 47,638 (2.64%)
  INFORMATION TECHNOLOGY: 10,455 (0.58%)
  COMPUTER ENGINEERING: 10,382 (0.58%)
  ELECTRICAL ENGINEERING: 9,714 (0.54%)
  MECHANICAL ENGINEERING: 9,200 (0.51%)

【Describe】
<class 'pandas.core.series.Series'>
RangeIndex: 1804286 entries, 0 to 1804285
Series name: BEN_PFIELD_OF_STUDY
Non-Null Count   Dtype 
--------------   ----- 
376319 non-null  object
dtypes: object(1)
memory usage: 13.8+ MB


In [57]:
# 6. WAGE_AMT
print("WAGE_AMT")

# Datatype
print(f"\n【Datatype】")
print(f"Type: {df_all['WAGE_AMT'].dtype}")
print(f"Classification: Quantitative/Numerical")

# Basic statistics
print(f"\n【Basic statistics】")
print(f"Total rows: {len(df_all):,}")
print(f"Non-null values: {df_all['WAGE_AMT'].notna().sum():,}")
print(f"Missing values: {df_all['WAGE_AMT'].isna().sum():,}")
print(f"Missing portion: {df_all['WAGE_AMT'].isna().sum() / len(df_all) * 100:.2f}%")
print(f"Unique values: {df_all['WAGE_AMT'].nunique():,}")

# 转换为数值类型
wage_numeric = pd.to_numeric(df_all['WAGE_AMT'], errors='coerce')

# Quantitative statistics
print(f"\n【Quantitative Statistics】")
if wage_numeric.notna().sum() > 0:
    print(f"Mean: {wage_numeric.mean():,.2f}")
    print(f"Median: {wage_numeric.median():,.2f}")
    mode_values = wage_numeric.mode()
    if len(mode_values) > 0:
        print(f"Mode: {mode_values.iloc[0]:,.2f}")
    print(f"Standard Deviation: {wage_numeric.std():,.2f}")
    print(f"Min: {wage_numeric.min():,.2f}")
    print(f"Max: {wage_numeric.max():,.2f}")
    print(f"Range: {wage_numeric.max() - wage_numeric.min():,.2f}")
    print(f"25th Percentile: {wage_numeric.quantile(0.25):,.2f}")
    print(f"75th Percentile: {wage_numeric.quantile(0.75):,.2f}")

# Top 5 values
print(f"\n【Top 5 values】")
top5 = wage_numeric.value_counts().head(5)
for value, count in top5.items():
    percentage = count / len(df_all) * 100
    print(f"  {value:,.2f}: {count:,} ({percentage:.2f}%)")

# Describe
print(f"\n【Describe】")
wage_numeric.describe()
df_all['WAGE_AMT'].info()

WAGE_AMT

【Datatype】
Type: object
Classification: Quantitative/Numerical

【Basic statistics】
Total rows: 1,804,286
Non-null values: 186,275
Missing values: 1,618,011
Missing portion: 89.68%
Unique values: 31,209

【Quantitative Statistics】
Mean: 142,844.77
Median: 94,037.00
Mode: 90,000.00
Standard Deviation: 15,268,608.51
Min: 0.00
Max: 6,585,365,853.00
Range: 6,585,365,853.00
25th Percentile: 78,000.00
75th Percentile: 120,000.00

【Top 5 values】
  90,000.00: 3,744 (0.21%)
  80,000.00: 3,394 (0.19%)
  100,000.00: 3,067 (0.17%)
  95,000.00: 3,015 (0.17%)
  85,000.00: 2,755 (0.15%)

【Describe】
<class 'pandas.core.series.Series'>
RangeIndex: 1804286 entries, 0 to 1804285
Series name: WAGE_AMT
Non-Null Count   Dtype 
--------------   ----- 
186275 non-null  object
dtypes: object(1)
memory usage: 13.8+ MB
